# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

## Part 1 answers (PySpark)

I did the main lab with the DataFrame API. Part 3 redoes some questions in SQL.

In [5]:
import calendar
import datetime
import math
import os

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# The timestamps are timestamp_ntz (local New York time without timezone). A fixed UTC session
# timezone makes the date/hour functions give the same results whatever the machine timezone is.
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_trips.printSchema()
print(f"{df_trips.count():,} records in the January 2019 file")

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)



7,696,617 records in the January 2019 file


### Unique key for each trip

monotonically_increasing_id() gives each row a unique 64-bit id. The ids increase but are not consecutive, and they depend on the partitioning, so I cache the DataFrame to keep them stable. The same cell adds the columns used later (date, hour, weekday, duration).

In [6]:
def add_trip_features(df):
    """Add the calendar and duration columns used throughout the lab."""
    return (df
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        # dayofweek: 1 = Sunday, 2 = Monday, ..., 7 = Saturday
        .withColumn("pickup_dow", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("pickup_day_name", F.date_format("tpep_pickup_datetime", "EEEE"))
        .withColumn("duration_min",
                    F.expr("timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)") / 60)
        # Spark 4 runs in ANSI mode: a division by 0 raises an error, so the speed is only
        # computed when the duration is positive (NULL otherwise)
        .withColumn("speed_mph", F.when(F.col("duration_min") > 0,
                                        F.col("trip_distance") / (F.col("duration_min") / 60))))


df_trips = add_trip_features(
    df_trips.withColumn("trip_id", F.monotonically_increasing_id())
).cache()

n_rows = df_trips.count()
n_ids = df_trips.select("trip_id").distinct().count()
print(f"rows: {n_rows:,} | distinct trip_id: {n_ids:,}")
assert n_rows == n_ids, "trip_id is not unique"

df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
                "duration_min", "pickup_day_name", "pickup_hour").show(5)

rows: 7,696,617 | distinct trip_id: 7,696,617
+-----------+--------------------+---------------------+------------------+---------------+-----------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|      duration_min|pickup_day_name|pickup_hour|
+-----------+--------------------+---------------------+------------------+---------------+-----------+
|68719476736| 2019-01-01 00:46:40|  2019-01-01 00:53:20| 6.666666666666667|        Tuesday|          0|
|68719476737| 2019-01-01 00:59:47|  2019-01-01 01:18:59|              19.2|        Tuesday|          0|
|68719476738| 2018-12-21 13:48:30|  2018-12-21 13:52:40| 4.166666666666667|         Friday|         13|
|68719476739| 2018-11-28 15:52:25|  2018-11-28 15:55:45|3.3333333333333335|      Wednesday|         15|
|68719476740| 2018-11-28 15:56:57|  2018-11-28 15:58:33|               1.6|      Wednesday|         15|
+-----------+--------------------+---------------------+------------------+---------------+-----------+
only showing top 5

### Valid trips

The raw file has records that can't be real trips (pickups in 2088, 30-day trips, a \$623,259 fare…). For the statistics I use df_valid, which keeps a trip if the pickup is in January 2019, it lasts between 1 minute and 3 hours, the distance is between 0 and 100 miles, the average speed is under 80 mph and the fare is between \$0 and \$500. Passenger count isn't in the rules, since a missing value doesn't mean the trip didn't happen. Questions about one specific record are also answered on the raw data.

In [7]:
def valid_trips(df, year, month):
    """Keep the records that look like real trips of the given month."""
    start = datetime.date(year, month, 1)
    end = datetime.date(year + month // 12, month % 12 + 1, 1)
    return df.filter(
        (F.col("pickup_date") >= F.lit(start)) & (F.col("pickup_date") < F.lit(end))
        & F.col("duration_min").between(1, 180)
        & (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)
        & (F.col("speed_mph") <= 80)
        & (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500))


def month_calendar(year, month):
    """One row per day of the month, used to average per weekday."""
    start = datetime.date(year, month, 1)
    end = datetime.date(year, month, calendar.monthrange(year, month)[1])
    return (spark.range(1)
        .select(F.explode(F.sequence(F.lit(start), F.lit(end))).alias("pickup_date"))
        .withColumn("pickup_dow", F.dayofweek("pickup_date"))
        .withColumn("pickup_day_name", F.date_format("pickup_date", "EEEE")))


df_valid = valid_trips(df_trips, 2019, 1).cache()
n_valid = df_valid.count()
print(f"valid trips: {n_valid:,} / {n_rows:,} ({n_valid / n_rows:.2%})")

valid trips: 7,583,742 / 7,696,617 (98.53%)


### Which trip has the highest passenger count

In [8]:
max_passengers = df_trips.agg(F.max("passenger_count")).first()[0]
df_max_passengers = df_trips.filter(F.col("passenger_count") == max_passengers)
print(f"highest passenger count: {max_passengers:.0f}, shared by {df_max_passengers.count()} trips")

df_max_passengers.select("trip_id", "tpep_pickup_datetime", "passenger_count", "trip_distance",
                         "duration_min", "fare_amount", "total_amount", "payment_type").show()

highest passenger count: 9, shared by 9 trips


+-----------+--------------------+---------------+-------------+-------------------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|passenger_count|trip_distance|       duration_min|fare_amount|total_amount|payment_type|
+-----------+--------------------+---------------+-------------+-------------------+-----------+------------+------------+
|68720426692| 2019-01-05 13:12:29|            9.0|          0.0|               0.05|        9.8|        12.6|           1|
|68720773023| 2019-01-07 03:19:36|            9.0|          0.0| 0.4166666666666667|        9.0|         9.3|           1|
|68721488834| 2019-01-10 00:43:10|            9.0|          0.0|0.06666666666666667|        9.0|        11.3|           1|
|68722360731| 2019-01-13 04:13:24|            9.0|          0.0| 1.1666666666666667|        9.0|       12.25|           1|
|68724011443| 2019-01-19 16:45:25|            9.0|          0.0|0.03333333333333333|       92.0|      110.76|           1|
|68724328961| 20

In [9]:
# Passenger count distribution, to put the maximum in perspective
df_trips.groupBy("passenger_count").count().orderBy("passenger_count").show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|  28672|
|            0.0| 117381|
|            1.0|5456515|
|            2.0|1113894|
|            3.0| 314692|
|            4.0| 140753|
|            5.0| 323842|
|            6.0| 200811|
|            7.0|     19|
|            8.0|     29|
|            9.0|      9|
+---------------+-------+



The max is 9, and 9 different trips have it. They're very likely errors: a yellow cab can take at most 5 passengers plus a child on a lap, and 8 of these 9 trips cover 0 miles in less than 1.5 minute. Only 19, 29 and 9 trips have 7, 8 and 9 passengers.

### What is the average passenger count

In [10]:
df_trips.agg(
    F.avg("passenger_count").alias("avg_all_non_null"),
    F.avg(F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count")))
        .alias("avg_1_to_6_passengers"),
).show()

df_valid.agg(
    F.avg(F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count")))
        .alias("avg_valid_trips_1_to_6_passengers")
).show()

+------------------+---------------------+
|  avg_all_non_null|avg_1_to_6_passengers|
+------------------+---------------------+
|1.5670317144945614|    1.591345720227794|
+------------------+---------------------+



+---------------------------------+
|avg_valid_trips_1_to_6_passengers|
+---------------------------------+
|               1.5924153189677637|
+---------------------------------+



1.57 over the records that have a value (avg skips the 28,672 nulls). A trip with 0 passengers isn't realistic, so keeping only 1 to 6 gives 1.59. Most rides are solo: 5.46M of the 7.7M trips.

### Shortest / longest trip by distance and by time

In [11]:
trip_cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
             "duration_min", "speed_mph", "fare_amount", "RatecodeID", "PULocationID", "DOLocationID"]

print("RAW DATA - longest by distance")
df_trips.orderBy(F.desc("trip_distance")).select(trip_cols).show(3)
print("RAW DATA - longest by time")
df_trips.orderBy(F.desc("duration_min")).select(trip_cols).show(3)
print("RAW DATA - shortest by time")
df_trips.orderBy("duration_min").select(trip_cols).show(3)
print("RAW DATA - trips with distance = 0:", df_trips.filter(F.col("trip_distance") == 0).count())

RAW DATA - longest by distance


+-----------+--------------------+---------------------+-------------+-----------------+------------------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|     duration_min|         speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+-----------------+------------------+-----------+----------+------------+------------+
|68725550827| 2019-01-25 21:56:39|  2019-01-25 22:06:08|        831.8|9.483333333333333| 5262.706502636204|        8.5|       1.0|         140|         239|
|68723763369| 2019-01-18 16:32:24|  2019-01-18 16:39:20|        700.7|6.933333333333334|           6063.75|        6.0|       1.0|         236|         262|
|68726247721| 2019-01-28 17:24:11|  2019-01-29 04:37:16|       214.01|673.0833333333334|19.077281168750773|      760.0|       1.0|         265|         265|
+-----------+--------------------+---------------------+--

+-----------+--------------------+---------------------+-------------+------------------+--------------------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|           speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+--------------------+-----------+----------+------------+------------+
|68719545003| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2| 43648.01666666667|0.001649559487429...|        6.5|       1.0|          48|         163|
|68720068998| 2019-01-03 22:24:36|  2019-01-27 10:41:17|          1.1|33856.683333333334|0.001949393546621...|        8.5|       1.0|          50|         170|
|68720352592| 2019-01-05 04:21:40|  2019-01-27 01:53:46|          3.9|           31532.1|0.007421009066950822|       16.5|       1.0|         114|         237|
+-----------+--------------------+------

+-----------+--------------------+---------------------+-------------+-------------------+---------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|       duration_min|speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+-------------------+---------+-----------+----------+------------+------------+
|68720679920| 2019-01-06 15:15:08|  2018-11-09 02:34:38|          3.3|           -84280.5|     NULL|       16.0|       1.0|         100|          79|
|68722453029| 2019-01-13 15:15:27|  2018-12-25 07:37:43|          7.6|-27817.733333333334|     NULL|       29.5|       1.0|         164|         145|
|68724218482| 2019-01-20 15:15:30|  2019-01-18 15:57:16|          1.9| -2838.233333333333|     NULL|       15.5|       1.0|         186|         162|
+-----------+--------------------+---------------------+-------------+-------------------+---------+

In [12]:
print("VALID TRIPS - longest by distance")
df_valid.orderBy(F.desc("trip_distance")).select(trip_cols).show(3)

print("VALID TRIPS - longest by time")
df_valid.orderBy(F.desc("duration_min")).select(trip_cols).show(3)

min_distance = df_valid.agg(F.min("trip_distance")).first()[0]
print(f"VALID TRIPS - shortest by distance: {min_distance} mi, "
      f"shared by {df_valid.filter(F.col('trip_distance') == min_distance).count():,} trips")
df_valid.filter(F.col("trip_distance") == min_distance).orderBy("duration_min").select(trip_cols).show(3)

min_duration = df_valid.agg(F.min("duration_min")).first()[0]
print(f"VALID TRIPS - shortest by time: {min_duration} min, "
      f"shared by {df_valid.filter(F.col('duration_min') == min_duration).count():,} trips")
df_valid.filter(F.col("duration_min") == min_duration).orderBy(F.desc("trip_distance")).select(trip_cols).show(3)

VALID TRIPS - longest by distance


+-----------+--------------------+---------------------+-------------+------------------+-----------------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|        speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+-----------------+-----------+----------+------------+------------+
|68722478266| 2019-01-13 17:40:12|  2019-01-13 20:02:32|        98.38|142.33333333333334|41.47166276346604|      300.0|       5.0|         215|         265|
|68724139852| 2019-01-20 03:44:56|  2019-01-20 05:18:54|        96.13| 93.96666666666667|61.38134090102873|      200.0|       5.0|         100|         265|
|68723700327| 2019-01-18 11:34:58|  2019-01-18 13:10:34|        91.86|              95.6|57.65271966527197|      240.0|       5.0|         132|         265|
+-----------+--------------------+---------------------+--

+-----------+--------------------+---------------------+-------------+------------------+-------------------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|          speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+-------------------+-----------+----------+------------+------------+
|68721731384| 2019-01-10 21:00:00|  2019-01-11 00:00:00|         3.12|             180.0|               1.04|       14.0|       1.0|         237|          68|
|68722008354| 2019-01-11 21:00:28|  2019-01-12 00:00:00|         1.34|179.53333333333333|0.44782770144819906|        9.5|       1.0|         163|         143|
|68719935800| 2019-01-03 11:56:30|  2019-01-03 14:55:56|         4.43|179.43333333333334|   1.48133011331971|       52.0|       2.0|         143|         264|
+-----------+--------------------+------------

+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+
|68719734703| 2019-01-02 12:50:55|  2019-01-02 12:51:55|         0.01|         1.0|      0.6|        2.5|       1.0|         114|         114|
|68719815020| 2019-01-02 18:12:52|  2019-01-02 18:13:52|         0.01|         1.0|      0.6|       52.0|       2.0|         132|         132|
|68720557297| 2019-01-05 23:19:01|  2019-01-05 23:20:01|         0.01|         1.0|      0.6|        2.5|       1.0|         107|         107|
+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+

+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|speed_mph|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+
|68721697876| 2019-01-10 19:37:16|  2019-01-10 19:38:16|          1.3|         1.0|     78.0|        8.8|       5.0|          79|          79|
|68722360049| 2019-01-13 04:29:38|  2019-01-13 04:30:38|          1.3|         1.0|     78.0|       80.0|       5.0|         265|         265|
|68725676036| 2019-01-26 11:44:46|  2019-01-26 11:45:46|          1.3|         1.0|     78.0|        2.5|       1.0|         264|         264|
+-----------+--------------------+---------------------+-------------+------------+---------+-----------+----------+------------+------------+

On the raw data the extremes are all nonsense: 831.8 miles in 9.5 minutes, a 30-day trip, negative durations (a dropoff two months before the pickup). On valid trips, the longest by distance is 98.38 miles (a \$300 negotiated fare to outside NYC). The longest by time hits my 3 h limit, with a dropoff at exactly midnight, so probably a meter left on. The shortest are 0.01 mile (638 trips) and 1 minute (907 trips), both at the edge of my rules, so these answers depend a lot on the cleaning.

### Busiest / slowest single day

In [13]:
df_daily = (df_valid
    .groupBy("pickup_date", "pickup_day_name")
    .count()
    .withColumnRenamed("count", "trips"))

print("Busiest days")
df_daily.orderBy(F.desc("trips")).show(5)
print("Slowest days")
df_daily.orderBy("trips").show(5)

Busiest days


+-----------+---------------+------+
|pickup_date|pickup_day_name| trips|
+-----------+---------------+------+
| 2019-01-25|         Friday|288452|
| 2019-01-11|         Friday|287562|
| 2019-01-17|       Thursday|280666|
| 2019-01-31|       Thursday|280464|
| 2019-01-24|       Thursday|278264|
+-----------+---------------+------+
only showing top 5 rows
Slowest days


+-----------+---------------+------+
|pickup_date|pickup_day_name| trips|
+-----------+---------------+------+
| 2019-01-01|        Tuesday|185715|
| 2019-01-21|         Monday|189297|
| 2019-01-02|      Wednesday|195419|
| 2019-01-20|         Sunday|199782|
| 2019-01-06|         Sunday|205378|
+-----------+---------------+------+
only showing top 5 rows


Busiest day: Friday 25 January with 288,452 trips (Friday 11 January is right behind). Slowest: 1 January with 185,715, which makes sense for New Year's Day. Next come 21 January (Martin Luther King Jr. Day) and 2 January.

### Busiest / slowest time of day

I count trips by pickup hour, then by period. The periods don't have the same length, so I compare them with trips per hour.

In [14]:
n_days = df_valid.select("pickup_date").distinct().count()

df_hourly = (df_valid
    .groupBy("pickup_hour")
    .count()
    .withColumnRenamed("count", "trips")
    .withColumn("avg_trips_per_day", F.round(F.col("trips") / n_days, 0))
    .orderBy("pickup_hour"))
df_hourly.show(24)

+-----------+------+-----------------+
|pickup_hour| trips|avg_trips_per_day|
+-----------+------+-----------------+
|          0|203863|           6576.0|
|          1|146239|           4717.0|
|          2|106861|           3447.0|
|          3| 75903|           2448.0|
|          4| 59290|           1913.0|
|          5| 73156|           2360.0|
|          6|174777|           5638.0|
|          7|299873|           9673.0|
|          8|368830|          11898.0|
|          9|361142|          11650.0|
|         10|356571|          11502.0|
|         11|370485|          11951.0|
|         12|395843|          12769.0|
|         13|398341|          12850.0|
|         14|426779|          13767.0|
|         15|445801|          14381.0|
|         16|414087|          13358.0|
|         17|462149|          14908.0|
|         18|509249|          16427.0|
|         19|469669|          15151.0|
|         20|418100|          13487.0|
|         21|404854|          13060.0|
|         22|364326|     

In [15]:
time_of_day = (F.when(F.col("pickup_hour").between(6, 11), "morning (06h-12h)")
    .when(F.col("pickup_hour").between(12, 16), "afternoon (12h-17h)")
    .when(F.col("pickup_hour").between(17, 21), "evening (17h-22h)")
    .otherwise("late night (22h-06h)"))

(df_valid
    .withColumn("time_of_day", time_of_day)
    .groupBy("time_of_day")
    .agg(F.count("*").alias("trips"), F.countDistinct("pickup_hour").alias("hours"))
    .withColumn("avg_trips_per_hour", F.round(F.col("trips") / (F.col("hours") * n_days), 0))
    .orderBy(F.desc("avg_trips_per_hour"))
    .show(truncate=False))

+--------------------+-------+-----+------------------+
|time_of_day         |trips  |hours|avg_trips_per_hour|
+--------------------+-------+-----+------------------+
|evening (17h-22h)   |2264021|5    |14607.0           |
|afternoon (12h-17h) |2080851|5    |13425.0           |
|morning (06h-12h)   |1931678|6    |10385.0           |
|late night (22h-06h)|1307192|8    |5271.0            |
+--------------------+-------+-----+------------------+



The peak is 18h-19h with about 16,400 trips per day, the low is 4h-5h with about 1,900, so 8.6 times fewer. By period, the evening (17h-22h) is the busiest and late night (22h-6h) the quietest, with morning and afternoon in between.

### On average, which day of the week is slowest / busiest

January 2019 has 5 Tuesdays, Wednesdays and Thursdays but only 4 of the other days, so I average the daily totals per weekday instead of just counting.

In [16]:
df_dow_avg = (month_calendar(2019, 1)
    .join(df_daily.drop("pickup_day_name"), "pickup_date", "left")
    .fillna(0, subset=["trips"])
    .groupBy("pickup_dow", "pickup_day_name")
    .agg(F.count("*").alias("days_in_month"), F.round(F.avg("trips"), 0).alias("avg_trips_per_day"))
    .orderBy(F.desc("avg_trips_per_day")))
df_dow_avg.show()

+----------+---------------+-------------+-----------------+
|pickup_dow|pickup_day_name|days_in_month|avg_trips_per_day|
+----------+---------------+-------------+-----------------+
|         6|         Friday|            4|         267922.0|
|         5|       Thursday|            5|         267587.0|
|         4|      Wednesday|            5|         249507.0|
|         7|       Saturday|            4|         248781.0|
|         3|        Tuesday|            5|         238261.0|
|         2|         Monday|            4|         223551.0|
|         1|         Sunday|            4|         211488.0|
+----------+---------------+-------------+-----------------+



Friday is the busiest on average (267,922 trips per day), just ahead of Thursday. Sunday is the slowest (211,488). With raw counts Thursday would have won, only because there are 5 of them.

### Does trip distance or the number of passengers affect the tip amount

I only use card payments (payment_type = 1), since cash tips are almost never recorded.

In [17]:
df_card = (df_valid
    .filter(F.col("payment_type") == 1)
    .withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount") * 100))

df_cash = df_valid.filter(F.col("payment_type") == 2)
print(f"cash trips: {df_cash.count():,}, with a recorded tip: {df_cash.filter(F.col('tip_amount') > 0).count():,}")

df_card.select(
    F.round(F.corr("trip_distance", "tip_amount"), 3).alias("corr(distance, tip)"),
    F.round(F.corr("passenger_count", "tip_amount"), 3).alias("corr(passengers, tip)"),
    F.round(F.corr("trip_distance", "tip_pct"), 3).alias("corr(distance, tip %)"),
).show()

cash trips: 2,089,812, with a recorded tip: 309


+-------------------+---------------------+---------------------+
|corr(distance, tip)|corr(passengers, tip)|corr(distance, tip %)|
+-------------------+---------------------+---------------------+
|              0.719|                0.011|               -0.007|
+-------------------+---------------------+---------------------+



In [18]:
distance_bucket = (F.when(F.col("trip_distance") < 1, "a. < 1 mi")
    .when(F.col("trip_distance") < 2, "b. 1-2 mi")
    .when(F.col("trip_distance") < 5, "c. 2-5 mi")
    .when(F.col("trip_distance") < 10, "d. 5-10 mi")
    .when(F.col("trip_distance") < 20, "e. 10-20 mi")
    .otherwise("f. >= 20 mi"))

(df_card
    .groupBy(distance_bucket.alias("distance"))
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
         F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct_of_fare"))
    .orderBy("distance")
    .show())

(df_card
    .filter(F.col("passenger_count").between(1, 6))
    .groupBy("passenger_count")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
         F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct_of_fare"),
         F.round(F.avg("trip_distance"), 2).alias("avg_distance"))
    .orderBy("passenger_count")
    .show())

+-----------+-------+-------+-------------------+
|   distance|  trips|avg_tip|avg_tip_pct_of_fare|
+-----------+-------+-------+-------------------+
|  a. < 1 mi|1362208|   1.32|               24.6|
|  b. 1-2 mi|1892864|   1.81|               21.9|
|  c. 2-5 mi|1425203|   2.66|               20.4|
| d. 5-10 mi| 431896|   4.68|               19.1|
|e. 10-20 mi| 288303|   8.11|               18.5|
|f. >= 20 mi|  36365|  10.53|               17.2|
+-----------+-------+-------+-------------------+



+---------------+-------+-------+-------------------+------------+
|passenger_count|  trips|avg_tip|avg_tip_pct_of_fare|avg_distance|
+---------------+-------+-------+-------------------+------------+
|            1.0|3901121|   2.51|               21.7|        2.92|
|            2.0| 774566|   2.59|               21.7|        2.99|
|            3.0| 216736|   2.56|               21.8|        2.92|
|            4.0|  91159|   2.57|               22.0|        2.91|
|            5.0| 229267|    2.6|               22.0|        2.97|
|            6.0| 141790|   2.59|               22.0|        2.96|
+---------------+-------+-------+-------------------+------------+



Distance, yes: the correlation is 0.72 and the average tip goes from \$1.32 under 1 mile to \$10.53 over 20 miles. That's mostly because the tip is a percentage of the fare, and the fare grows with distance. As a percentage it actually drops a bit (24.6% down to 17.2%). Passengers, no: the correlation is 0.01 and the tip stays around \$2.5 whatever the group size.

### What was the highest "extra" charge and which trip

In [19]:
extra_cols = ["trip_id", "tpep_pickup_datetime", "trip_distance", "duration_min", "fare_amount",
              "extra", "total_amount", "RatecodeID", "PULocationID", "DOLocationID"]

print("RAW DATA")
df_trips.orderBy(F.desc("extra")).select(extra_cols).show(3)

max_extra = df_valid.agg(F.max("extra")).first()[0]
print(f"VALID TRIPS - highest extra: {max_extra}")
df_valid.filter(F.col("extra") == max_extra).select(extra_cols).show()

RAW DATA


+-----------+--------------------+-------------+-----------------+-----------+------+------------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|     duration_min|fare_amount| extra|total_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+-------------+-----------------+-----------+------+------------+----------+------------+------------+
|68724800219| 2019-01-23 08:58:09|          0.0|              0.0|  355676.98|535.38|   356214.78|       1.0|          24|         264|
|68726929966| 2019-01-31 10:06:09|          0.0|              0.0|        4.5| 23.04|       28.34|       1.0|         237|         264|
|68720019939| 2019-01-03 18:32:36|         16.6|72.88333333333334|       61.0|  18.5|        92.3|       3.0|         158|           1|
+-----------+--------------------+-------------+-----------------+-----------+------+------------+----------+------------+------------+
only showing top 3 rows
VALID TRIPS - highest ex

In [20]:
# Where do the large extras come from? RatecodeID 3 = Newark, LocationID 1 = Newark Airport (EWR)
(df_valid
    .filter(F.col("extra") >= 10)
    .groupBy("extra", "RatecodeID", "DOLocationID")
    .count()
    .orderBy(F.desc("extra"), F.desc("count"))
    .show(10))

+-----+----------+------------+-----+
|extra|RatecodeID|DOLocationID|count|
+-----+----------+------------+-----+
| 18.5|       3.0|           1|    6|
| 18.0|       3.0|           1|    9|
| 17.5|       3.0|           1|   63|
| 10.9|       1.0|         230|    1|
+-----+----------+------------+-----+



Raw data: \$535.38, on a record with 0 miles, 0 seconds and a \$355,676.98 fare, so clearly an error. Valid trips: \$18.50, for 6 trips, all to Newark Airport between 16h and 19h on weekdays. That seems to be the \$17.50 Newark surcharge plus the \$1 rush-hour extra.

### Are there any datapoints that seem to be strange / outliers

In [21]:
checks = {
    "pickup outside January 2019": ~F.col("pickup_date").between(datetime.date(2019, 1, 1), datetime.date(2019, 1, 31)),
    "dropoff before or at pickup (duration <= 0)": F.col("duration_min") <= 0,
    "duration < 1 min": (F.col("duration_min") > 0) & (F.col("duration_min") < 1),
    "duration > 3 h": F.col("duration_min") > 180,
    "duration > 23 h": F.col("duration_min") > 23 * 60,
    "distance = 0": F.col("trip_distance") == 0,
    "distance > 100 mi": F.col("trip_distance") > 100,
    "speed > 80 mph": F.col("speed_mph") > 80,
    "fare < 0": F.col("fare_amount") < 0,
    "fare = 0": F.col("fare_amount") == 0,
    "fare > $500": F.col("fare_amount") > 500,
    "passenger_count null": F.col("passenger_count").isNull(),
    "passenger_count = 0": F.col("passenger_count") == 0,
    "passenger_count > 6": F.col("passenger_count") > 6,
    "tip < 0": F.col("tip_amount") < 0,
    "unknown pickup zone (264/265)": F.col("PULocationID").isin(264, 265),
}

# one pass over the data: sum a 0/1 flag per check
df_checks = df_trips.agg(*[
    F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in checks.items()
]).first().asDict()

for name, count in df_checks.items():
    print(f"{name:<45} {count:>9,} ({count / n_rows:.3%})")
print(f"{'removed by the validity rules':<45} {n_rows - n_valid:>9,} ({(n_rows - n_valid) / n_rows:.3%})")

pickup outside January 2019                         537 (0.007%)
dropoff before or at pickup (duration <= 0)       6,557 (0.085%)
duration < 1 min                                 67,801 (0.881%)
duration > 3 h                                   20,914 (0.272%)
duration > 23 h                                  18,742 (0.244%)
distance = 0                                     55,089 (0.716%)
distance > 100 mi                                    32 (0.000%)
speed > 80 mph                                    6,147 (0.080%)
fare < 0                                          7,129 (0.093%)
fare = 0                                          2,641 (0.034%)
fare > $500                                          34 (0.000%)
passenger_count null                             28,672 (0.373%)
passenger_count = 0                             117,381 (1.525%)
passenger_count > 6                                  57 (0.001%)
tip < 0                                             105 (0.001%)
unknown pickup zone (264/

In [22]:
# Pickups outside January 2019, by year
(df_trips
    .filter(~F.col("pickup_date").between(datetime.date(2019, 1, 1), datetime.date(2019, 1, 31)))
    .groupBy(F.year("pickup_date").alias("year"))
    .count()
    .orderBy("year")
    .show())

# Durations close to 24 h: the meter was not switched off
(df_trips
    .filter(F.col("duration_min") > 180)
    .groupBy(F.floor(F.col("duration_min") / 60).alias("duration_hours"))
    .count()
    .orderBy(F.desc("count"))
    .show(5))

# Records with a negative fare: the other amounts are negative too, it is a refund / void
(df_trips
    .filter(F.col("fare_amount") < 0)
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"), F.round(F.avg("total_amount"), 2).alias("avg_total"))
    .orderBy(F.desc("trips"))
    .show())

+----+-----+
|year|count|
+----+-----+
|2001|    1|
|2003|    2|
|2008|   22|
|2009|   50|
|2018|  366|
|2019|   94|
|2088|    2|
+----+-----+

+--------------+-----+
|duration_hours|count|
+--------------+-----+
|            23|18739|
|            22|  497|
|             3|  148|
|             4|  136|
|             8|  122|
+--------------+-----+
only showing top 5 rows


+------------+-----+---------+
|payment_type|trips|avg_total|
+------------+-----+---------+
|           3| 4083|    -9.76|
|           4| 2667|   -11.96|
|           2|  376|   -21.32|
|           0|    2|      0.0|
|           1|    1|    -12.3|
+------------+-----+---------+



Yes, about 1.5% of the records (112,875) break at least one of my rules. The dates first: 537 pickups aren't in January 2019, some in 2001 or even 2088, probably bad device clocks. Then durations: 6,557 trips end before they start and 18,739 last between 23 and 24 hours, which looks like meters nobody switched off. There are also 55,089 trips at 0 miles, 7,129 negative fares (refunds I think, the total is negative too) and a \$623,259.86 fare for 2.4 miles. For passengers, 117,381 trips have 0 and 28,672 have no value; I'd guess the driver just didn't enter it. Spark reads all of this without complaining because the types are fine, you only catch it with common sense.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

## Part 2 answers (PySpark)

### Load the taxi zone lookup

The lookup is a CSV, so I give the schema explicitly instead of inferring it.

In [23]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

zone_download_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_lookup_file = "taxi_zone_lookup.csv"

response = requests.get(zone_download_url)
if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)

zone_schema = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

df_zones = spark.read.csv(zone_lookup_file, header=True, schema=zone_schema)
print(f"{df_zones.count()} zones")
df_zones.groupBy("Borough").count().orderBy(F.desc("count")).show()

265 zones
+-------------+-----+
|      Borough|count|
+-------------+-----+
|       Queens|   69|
|    Manhattan|   69|
|     Brooklyn|   61|
|        Bronx|   43|
|Staten Island|   20|
|          EWR|    1|
|      Unknown|    1|
|          N/A|    1|
+-------------+-----+



Each trip is joined twice with the lookup, on pickup and on dropoff. The lookup only has 265 rows, so I broadcast it instead of shuffling the trips. By borough I mean the pickup borough unless I say otherwise.

In [24]:
def with_boroughs(df):
    """Add the pickup / dropoff borough and zone names."""
    pickup = df_zones.select(F.col("LocationID").alias("PULocationID"),
                             F.col("Borough").alias("pu_borough"), F.col("Zone").alias("pu_zone"))
    dropoff = df_zones.select(F.col("LocationID").alias("DOLocationID"),
                              F.col("Borough").alias("do_borough"), F.col("Zone").alias("do_zone"))
    return (df
        .join(F.broadcast(pickup), "PULocationID", "left")
        .join(F.broadcast(dropoff), "DOLocationID", "left"))


df_valid_boro = with_boroughs(df_valid).cache()
print("trips without a pickup borough:", df_valid_boro.filter(F.col("pu_borough").isNull()).count())

trips without a pickup borough: 0


### Which borough had the most pickups? dropoffs?

In [25]:
df_pickups = df_valid_boro.groupBy(F.col("pu_borough").alias("borough")).agg(F.count("*").alias("pickups"))
df_dropoffs = df_valid_boro.groupBy(F.col("do_borough").alias("borough")).agg(F.count("*").alias("dropoffs"))

(df_pickups
    .join(df_dropoffs, "borough", "full")
    .fillna(0)
    .withColumn("pickups_pct", F.round(F.col("pickups") / n_valid * 100, 2))
    .withColumn("dropoffs_pct", F.round(F.col("dropoffs") / n_valid * 100, 2))
    .orderBy(F.desc("pickups"))
    .show())

+-------------+-------+--------+-----------+------------+
|      borough|pickups|dropoffs|pickups_pct|dropoffs_pct|
+-------------+-------+--------+-----------+------------+
|    Manhattan|6876789| 6745222|      90.68|       88.94|
|       Queens| 450841|  320839|       5.94|        4.23|
|      Unknown| 149052|  138991|       1.97|        1.83|
|     Brooklyn|  88292|  295928|       1.16|         3.9|
|        Bronx|  16850|   56526|       0.22|        0.75|
|          N/A|   1591|   13850|       0.02|        0.18|
|Staten Island|    297|    2106|        0.0|        0.03|
|          EWR|     30|   10280|        0.0|        0.14|
+-------------+-------+--------+-----------+------------+



Manhattan, by far, for both: 90.7% of pickups and 88.9% of dropoffs. Queens comes second, mostly because of the airports. Brooklyn and the Bronx get about 3.4 times more dropoffs than pickups: people take a cab home from Manhattan but rarely find one to come back. Newark is extreme, with 10,280 dropoffs for 30 pickups.

### What are the busy / slow times by borough

Pickups per hour for each borough, then the busiest and slowest hour of each one with a window.

In [26]:
boroughs = ["Manhattan", "Queens", "Brooklyn", "Bronx", "Staten Island", "EWR", "Unknown", "N/A"]

(df_valid_boro
    .groupBy("pickup_hour")
    .pivot("pu_borough", boroughs)
    .count()
    .fillna(0)
    .orderBy("pickup_hour")
    .show(24))

+-----------+---------+------+--------+-----+-------------+---+-------+---+
|pickup_hour|Manhattan|Queens|Brooklyn|Bronx|Staten Island|EWR|Unknown|N/A|
+-----------+---------+------+--------+-----+-------------+---+-------+---+
|          0|   180685| 15580|    3676|  292|            3|  0|   3582| 45|
|          1|   132633|  7481|    3230|  227|            2|  0|   2628| 38|
|          2|    98606|  3497|    2511|  196|            3|  1|   2021| 26|
|          3|    69580|  2858|    1819|  194|            3|  0|   1420| 29|
|          4|    52223|  3711|    1825|  331|            3|  0|   1168| 29|
|          5|    61548|  7552|    1997|  643|            8|  2|   1375| 31|
|          6|   153561| 12249|    4260| 1206|           11|  1|   3431| 58|
|          7|   268996| 17299|    6073| 1690|           12|  2|   5707| 94|
|          8|   334986| 18523|    6720| 1380|           31|  1|   7097| 92|
|          9|   330289| 18299|    4503| 1116|           30|  2|   6802|101|
|         10

In [27]:
df_boro_hour = df_valid_boro.groupBy("pu_borough", "pickup_hour").agg(F.count("*").alias("trips"))

w_busy = Window.partitionBy("pu_borough").orderBy(F.desc("trips"), "pickup_hour")
w_slow = Window.partitionBy("pu_borough").orderBy("trips", "pickup_hour")

busiest_hour = (df_boro_hour
    .withColumn("rank", F.row_number().over(w_busy)).filter("rank = 1")
    .select("pu_borough", F.col("pickup_hour").alias("busiest_hour"), F.col("trips").alias("trips_busiest")))
slowest_hour = (df_boro_hour
    .withColumn("rank", F.row_number().over(w_slow)).filter("rank = 1")
    .select("pu_borough", F.col("pickup_hour").alias("slowest_hour"), F.col("trips").alias("trips_slowest")))

busiest_hour.join(slowest_hour, "pu_borough").orderBy(F.desc("trips_busiest")).show()

+-------------+------------+-------------+------------+-------------+
|   pu_borough|busiest_hour|trips_busiest|slowest_hour|trips_slowest|
+-------------+------------+-------------+------------+-------------+
|    Manhattan|          18|       467289|           4|        52223|
|       Queens|          21|        28654|           3|         2858|
|      Unknown|          18|        10203|           4|         1168|
|     Brooklyn|           8|         6720|           3|         1819|
|        Bronx|           7|         1690|           3|          194|
|          N/A|          14|          114|           2|           26|
|Staten Island|           8|           31|           1|            2|
|          EWR|          15|            6|           2|            1|
+-------------+------------+-------------+------------+-------------+



Manhattan follows the city curve and is busiest at 18h. Queens peaks at 21h but stays high from about 15h, probably the airport arrivals. Brooklyn and the Bronx are busiest in the morning (8h and 7h), people leaving home for work I guess. Everywhere the slowest hour is between 1h and 4h. Staten Island and EWR have too few trips to say much.

### What are the busiest days of the week by borough

Same method as in Part 1, per borough.

In [28]:
df_boro_daily = df_valid_boro.groupBy("pu_borough", "pickup_date").agg(F.count("*").alias("trips"))

df_boro_dow = (month_calendar(2019, 1)
    .crossJoin(df_valid_boro.select("pu_borough").distinct())
    .join(df_boro_daily, ["pu_borough", "pickup_date"], "left")
    .fillna(0, subset=["trips"])
    .groupBy("pu_borough", "pickup_dow", "pickup_day_name")
    .agg(F.round(F.avg("trips"), 1).alias("avg_trips")))

# average pickups per weekday, one column per borough
(df_boro_dow
    .groupBy("pickup_dow", "pickup_day_name")
    .pivot("pu_borough", boroughs)
    .agg(F.first("avg_trips"))
    .orderBy("pickup_dow")
    .show())

w_dow = Window.partitionBy("pu_borough").orderBy(F.desc("avg_trips"))
(df_boro_dow
    .withColumn("rank", F.row_number().over(w_dow))
    .filter("rank <= 2")
    .groupBy("pu_borough")
    .agg(F.concat_ws(", ", F.collect_list("pickup_day_name")).alias("two_busiest_days"))
    .orderBy("pu_borough")
    .show(truncate=False))

+----------+---------------+---------+-------+--------+-----+-------------+---+-------+----+
|pickup_dow|pickup_day_name|Manhattan| Queens|Brooklyn|Bronx|Staten Island|EWR|Unknown| N/A|
+----------+---------------+---------+-------+--------+-----+-------------+---+-------+----+
|         1|         Sunday| 190237.8|14216.8|  2654.5|494.0|          8.5|1.0| 3829.5|45.8|
|         2|         Monday| 199717.5|15953.0|  2289.3|507.0|          9.8|0.0| 5019.8|54.5|
|         3|        Tuesday| 214930.6|15081.4|  3040.2|574.2|          8.4|0.6| 4568.2|57.6|
|         4|      Wednesday| 226686.4|14484.4|  2896.4|560.4|          7.2|1.8| 4813.8|56.8|
|         5|       Thursday| 243419.2|15080.2|  3016.0|579.2|         10.2|1.2| 5428.4|52.4|
|         6|         Friday| 243677.0|15331.5|  3150.3|623.3|         14.5|1.0| 5079.0|45.8|
|         7|       Saturday| 229269.8|11401.5|  2788.3|446.0|          9.3|1.0| 4821.8|43.3|
+----------+---------------+---------+-------+--------+-----+---------

+-------------+-------------------+
|pu_borough   |two_busiest_days   |
+-------------+-------------------+
|Bronx        |Friday, Thursday   |
|Brooklyn     |Friday, Tuesday    |
|EWR          |Wednesday, Thursday|
|Manhattan    |Friday, Thursday   |
|N/A          |Tuesday, Wednesday |
|Queens       |Monday, Friday     |
|Staten Island|Friday, Thursday   |
|Unknown      |Thursday, Friday   |
+-------------+-------------------+



Friday comes out on top almost everywhere (Manhattan, Brooklyn, Bronx), with Thursday close behind in Manhattan. Queens is different: Monday and Friday, which looks like airport traffic around weekends (Monday 21 January was also a holiday). Staten Island and EWR have too few pickups to rank.

### What is the average trip distance and the average trip fare by borough

In [29]:
df_boro_avg = (df_valid_boro
    .groupBy("pu_borough")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("total_amount"), 2).alias("avg_total_amount"))
    .orderBy(F.desc("trips")))
df_boro_avg.show()

+-------------+-------+---------------+--------+----------------+
|   pu_borough|  trips|avg_distance_mi|avg_fare|avg_total_amount|
+-------------+-------+---------------+--------+----------------+
|    Manhattan|6876789|           2.24|   10.63|           13.49|
|       Queens| 450841|          11.69|   35.79|            45.3|
|      Unknown| 149052|           2.57|   11.53|           14.67|
|     Brooklyn|  88292|           4.94|   18.93|           21.93|
|        Bronx|  16850|           7.69|   27.22|           30.36|
|          N/A|   1591|           4.63|   30.54|           36.21|
|Staten Island|    297|          14.89|   47.69|           55.66|
|          EWR|     30|           8.56|   56.13|            72.4|
+-------------+-------+---------------+--------+----------------+



Manhattan trips are short and cheap: 2.24 miles and \$10.63 on average. Queens is at 11.69 miles and \$35.79, mostly JFK and LaGuardia rides with the \$52 JFK flat fare. Brooklyn (4.94 mi, \$18.93) and the Bronx (7.69 mi, \$27.22) are in between, and Staten Island has the longest trips (14.89 mi, \$47.69). Basically, the further from Manhattan, the longer and pricier.

### Highest / lowest fare amounts for a trip, and the associated borough

In [30]:
fare_cols = ["trip_id", "tpep_pickup_datetime", "trip_distance", "duration_min", "fare_amount",
             "RatecodeID", "payment_type", "pu_borough", "pu_zone", "do_borough", "do_zone"]

df_trips_boro = with_boroughs(df_trips)

print("RAW DATA - highest fares")
df_trips_boro.orderBy(F.desc("fare_amount")).select(fare_cols).show(3, truncate=False)
print("RAW DATA - lowest fares")
df_trips_boro.orderBy("fare_amount").select(fare_cols).show(3, truncate=False)

RAW DATA - highest fares


+-----------+--------------------+-------------+------------+-----------+----------+------------+----------+---------------------+----------+--------+
|trip_id    |tpep_pickup_datetime|trip_distance|duration_min|fare_amount|RatecodeID|payment_type|pu_borough|pu_zone              |do_borough|do_zone |
+-----------+--------------------+-------------+------------+-----------+----------+------------+----------+---------------------+----------+--------+
|68721976391|2019-01-11 19:33:15 |2.4          |19.9        |623259.86  |1.0       |3           |Manhattan |Upper East Side South|Manhattan |Flatiron|
|68724800219|2019-01-23 08:58:09 |0.0          |0.0         |355676.98  |1.0       |2           |Manhattan |Bloomingdale         |Unknown   |N/A     |
|68721636707|2019-01-10 16:08:10 |0.0          |0.0         |36090.3    |99.0      |1           |Unknown   |N/A                  |Unknown   |N/A     |
+-----------+--------------------+-------------+------------+-----------+----------+----------

+-----------+--------------------+-------------+-------------------+-----------+----------+------------+----------+---------------------------------+----------+---------------------------------+
|trip_id    |tpep_pickup_datetime|trip_distance|duration_min       |fare_amount|RatecodeID|payment_type|pu_borough|pu_zone                          |do_borough|do_zone                          |
+-----------+--------------------+-------------+-------------------+-----------+----------+------------+----------+---------------------------------+----------+---------------------------------+
|68724367385|2019-01-21 12:01:45 |0.0          |6.85               |-362.0     |5.0       |4           |Queens    |JFK Airport                      |Queens    |JFK Airport                      |
|68725784936|2019-01-26 18:46:42 |0.0          |0.13333333333333333|-320.0     |5.0       |3           |N/A       |Outside of NYC                   |N/A       |Outside of NYC                   |
|68719533830|2019-01-01 0

In [31]:
print("VALID TRIPS - highest fares")
df_valid_boro.orderBy(F.desc("fare_amount")).select(fare_cols).show(5, truncate=False)

min_fare = df_valid_boro.agg(F.min("fare_amount")).first()[0]
print(f"VALID TRIPS - lowest fare: {min_fare}")
(df_valid_boro
    .filter(F.col("fare_amount") == min_fare)
    .groupBy("pu_borough", "do_borough")
    .count()
    .orderBy(F.desc("count"))
    .show())

VALID TRIPS - highest fares


+-----------+--------------------+-------------+------------------+-----------+----------+------------+----------+--------------+----------+--------------+
|trip_id    |tpep_pickup_datetime|trip_distance|duration_min      |fare_amount|RatecodeID|payment_type|pu_borough|pu_zone       |do_borough|do_zone       |
+-----------+--------------------+-------------+------------------+-----------+----------+------------+----------+--------------+----------+--------------+
|68725013451|2019-01-23 22:17:37 |0.1          |1.2833333333333334|500.0      |5.0       |1           |N/A       |Outside of NYC|N/A       |Outside of NYC|
|68719753315|2019-01-02 13:48:37 |74.24        |114.55            |468.0      |5.0       |1           |Queens    |JFK Airport   |N/A       |Outside of NYC|
|68726430689|2019-01-29 13:19:20 |0.1          |1.7666666666666666|450.0      |5.0       |1           |Manhattan |Union Sq      |Manhattan |Midtown South |
|68720482476|2019-01-05 17:03:46 |67.78        |82.4666666666666

Raw data: the highest is \$623,259.86 for 2.4 miles in Manhattan (Upper East Side to Flatiron), obviously a typo. The lowest is -\$362, a refund at JFK Airport in Queens. On valid trips the top is \$500 for 0.1 mile outside NYC, which looks suspicious too; the first believable one is \$468 for 74 miles from JFK to outside NYC. The lowest is \$0.01, mostly in Manhattan and Queens. These extremes say more about data quality than about boroughs.

### Most recently available January: is there a change in the average metrics?

The notebook checks which January is the latest one online (January 2026 right now). I also load January 2025 since the instructions mention it. Every year gets the same cleaning.

In [32]:
def trip_data_url(year, month):
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"


def download(url, path):
    """Download a file once, streaming it to disk."""
    if not os.path.exists(path):
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
        os.rename(path + ".part", path)
    return path


def load_trips(year, month):
    path = download(trip_data_url(year, month), f"yellow_tripdata_{year}-{month:02d}.parquet")
    return spark.read.parquet(path)


latest_january = next(
    year for year in range(datetime.date.today().year, 2019, -1)
    if requests.head(trip_data_url(year, 1)).status_code == 200)
print("most recent January available:", latest_january)

compare_years = sorted({2025, latest_january})
df_valid_by_year = {2019: df_valid}
for year in compare_years:
    df_year = add_trip_features(load_trips(year, 1))
    df_valid_by_year[year] = valid_trips(df_year, year, 1).cache()
    print(f"January {year}: {df_year.count():,} records, {df_valid_by_year[year].count():,} valid trips")

most recent January available: 2026


January 2025: 3,475,226 records, 3,241,475 valid trips


January 2026: 3,724,889 records, 3,504,254 valid trips


In [33]:
def summary_metrics(df, year):
    card = F.col("payment_type") == 1
    return df.agg(
        F.count("*").alias("trips"),
        F.round(F.avg(F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))), 3)
            .alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg(F.when(card, F.col("tip_amount"))), 2).alias("avg_tip_card"),
        F.round(F.avg(F.when(card, F.col("tip_amount") / F.col("fare_amount") * 100)), 1)
            .alias("avg_tip_pct_card"),
        F.round(F.avg("total_amount"), 2).alias("avg_total"),
        # payment_type 0 = no payment information (these records also have no passenger count)
        F.round(F.avg(F.when(F.col("payment_type") > 0, card.cast("int"))) * 100, 1)
            .alias("card_pct_known_payment"),
        F.round(F.avg((F.col("payment_type") == 0).cast("int")) * 100, 1).alias("payment_type_0_pct"),
    ).withColumn("january", F.lit(year))


df_compare = None
for year, df in sorted(df_valid_by_year.items()):
    metrics = summary_metrics(df, year)
    df_compare = metrics if df_compare is None else df_compare.unionByName(metrics)

df_compare.select("january", *[c for c in df_compare.columns if c != "january"]).show()

+-------+-------+--------------+---------------+----------------+-------------+--------+------------+----------------+---------+----------------------+------------------+
|january|  trips|avg_passengers|avg_distance_mi|avg_duration_min|avg_speed_mph|avg_fare|avg_tip_card|avg_tip_pct_card|avg_total|card_pct_known_payment|payment_type_0_pct|
+-------+-------+--------------+---------------+----------------+-------------+--------+------------+----------------+---------+----------------------+------------------+
|   2019|7583742|         1.592|           2.85|           13.04|        11.73|   12.28|        2.53|            21.8|    15.55|                  72.0|               0.4|
|   2025|3241475|         1.306|           3.17|           14.72|        11.33|   17.91|        4.09|            26.1|    26.77|                  85.5|              12.7|
|   2026|3504254|          1.26|            3.5|           17.17|        11.12|   21.02|         4.1|            25.0|    29.61|                 

In [34]:
# Same comparison by pickup borough
df_boro_compare = None
for year, df in sorted(df_valid_by_year.items()):
    metrics = (with_boroughs(df)
        .filter(F.col("pu_borough").isin("Manhattan", "Queens", "Brooklyn", "Bronx"))
        .groupBy("pu_borough")
        .agg(F.count("*").alias("trips"),
             F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
             F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
        .withColumn("january", F.lit(year)))
    df_boro_compare = metrics if df_boro_compare is None else df_boro_compare.unionByName(metrics)

df_boro_compare.orderBy("pu_borough", "january").show()

+----------+-------+------------+--------+-------+
|pu_borough|  trips|avg_distance|avg_fare|january|
+----------+-------+------------+--------+-------+
|     Bronx|  16850|        7.69|   27.22|   2019|
|     Bronx|  12689|        7.55|   30.12|   2025|
|     Bronx|  34973|        7.78|   32.66|   2026|
|  Brooklyn|  88292|        4.94|   18.93|   2019|
|  Brooklyn|  55131|        5.45|   26.36|   2025|
|  Brooklyn| 145663|        5.77|   30.83|   2026|
| Manhattan|6876789|        2.24|   10.63|   2019|
| Manhattan|2899353|        2.27|   14.66|   2025|
| Manhattan|3002370|        2.53|   17.52|   2026|
|    Queens| 450841|       11.69|   35.79|   2019|
|    Queens| 266100|       12.27|   50.88|   2025|
|    Queens| 316239|       11.18|   48.32|   2026|
+----------+-------+------------+--------+-------+



Yes, a lot changed. There are less than half as many yellow trips (7.58M in 2019, 3.50M in 2026), probably because of Uber and Lyft. The average fare went from \$12.28 to \$21.02 and the total paid from \$15.55 to \$29.61, with the late-2022 fare increase and new surcharges like the congestion fee. Trips are a bit longer (2.85 to 3.50 miles) and slower. Card tips went from 21.8% to about 25% of the fare. One thing to watch: 28.3% of the 2026 trips have payment_type 0 and no passenger count, so I only average those fields where they exist.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

## Part 3 answers (Spark SQL)

Since the main part is in PySpark, I redid three questions in SQL and checked each result against the PySpark one: the average passenger count, the average trips per weekday, and the pickups, distance and fare by borough (the one with the join).

In [35]:
df_trips.createOrReplaceTempView("trips_raw")
df_valid.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

### 1. Average passenger count

In [36]:
df_sql_passengers = spark.sql("""
    SELECT
        AVG(passenger_count) AS avg_all_non_null,
        AVG(CASE WHEN passenger_count BETWEEN 1 AND 6 THEN passenger_count END) AS avg_1_to_6_passengers
    FROM trips_raw
""")
df_sql_passengers.show()

df_py_passengers = df_trips.agg(
    F.avg("passenger_count").alias("avg_all_non_null"),
    F.avg(F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))).alias("avg_1_to_6_passengers"))
sql_row, py_row = df_sql_passengers.first(), df_py_passengers.first()
assert all(math.isclose(a, b, rel_tol=1e-9) for a, b in zip(sql_row, py_row))
print("same result as PySpark")

+------------------+---------------------+
|  avg_all_non_null|avg_1_to_6_passengers|
+------------------+---------------------+
|1.5670317144945614|    1.591345720227794|
+------------------+---------------------+



same result as PySpark


### 2. Average number of trips per day of the week

In [37]:
df_sql_dow = spark.sql("""
    WITH month_days AS (
        SELECT explode(sequence(DATE'2019-01-01', DATE'2019-01-31')) AS pickup_date
    ),
    daily AS (
        SELECT pickup_date, COUNT(*) AS trips
        FROM trips
        GROUP BY pickup_date
    )
    SELECT
        dayofweek(d.pickup_date) AS pickup_dow,
        date_format(d.pickup_date, 'EEEE') AS pickup_day_name,
        COUNT(*) AS days_in_month,
        ROUND(AVG(COALESCE(t.trips, 0)), 0) AS avg_trips_per_day
    FROM month_days d
    LEFT JOIN daily t ON d.pickup_date = t.pickup_date
    GROUP BY 1, 2
    ORDER BY avg_trips_per_day DESC
""")
df_sql_dow.show()

assert df_sql_dow.collect() == df_dow_avg.collect()
print("same result as PySpark")

+----------+---------------+-------------+-----------------+
|pickup_dow|pickup_day_name|days_in_month|avg_trips_per_day|
+----------+---------------+-------------+-----------------+
|         6|         Friday|            4|         267922.0|
|         5|       Thursday|            5|         267587.0|
|         4|      Wednesday|            5|         249507.0|
|         7|       Saturday|            4|         248781.0|
|         3|        Tuesday|            5|         238261.0|
|         2|         Monday|            4|         223551.0|
|         1|         Sunday|            4|         211488.0|
+----------+---------------+-------------+-----------------+



same result as PySpark


### 3. Pickups, average distance and average fare by borough (join)

In [38]:
df_sql_boro = spark.sql("""
    SELECT
        z.Borough AS pu_borough,
        COUNT(*) AS trips,
        ROUND(AVG(t.trip_distance), 2) AS avg_distance_mi,
        ROUND(AVG(t.fare_amount), 2) AS avg_fare,
        ROUND(AVG(t.total_amount), 2) AS avg_total_amount
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY trips DESC
""")
df_sql_boro.show()

assert df_sql_boro.collect() == df_boro_avg.collect()
print("same result as PySpark")

+-------------+-------+---------------+--------+----------------+
|   pu_borough|  trips|avg_distance_mi|avg_fare|avg_total_amount|
+-------------+-------+---------------+--------+----------------+
|    Manhattan|6876789|           2.24|   10.63|           13.49|
|       Queens| 450841|          11.69|   35.79|            45.3|
|      Unknown| 149052|           2.57|   11.53|           14.67|
|     Brooklyn|  88292|           4.94|   18.93|           21.93|
|        Bronx|  16850|           7.69|   27.22|           30.36|
|          N/A|   1591|           4.63|   30.54|           36.21|
|Staten Island|    297|          14.89|   47.69|           55.66|
|          EWR|     30|           8.56|   56.13|            72.4|
+-------------+-------+---------------+--------+----------------+



same result as PySpark


Spark SQL and the DataFrame API go through the same optimizer (Catalyst), so they end up with the same plan. The explain below shows that the SQL join also becomes a broadcast join.

In [39]:
df_sql_boro.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- *(4) Sort [trips#52567L DESC NULLS LAST], true, 0
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 3
            +- Exchange rangepartitioning(trips#52567L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=9077]
               +- *(3) HashAggregate(keys=[Borough#27939], functions=[count(1), avg(trip_distance#4), avg(fare_amount#10), avg(total_amount#16)])
                  +- AQEShuffleRead coalesced
                     +- ShuffleQueryStage 2
                        +- Exchange hashpartitioning(Borough#27939, 200), ENSURE_REQUIREMENTS, [plan_id=9045]
                           +- *(2) HashAggregate(keys=[Borough#27939], functions=[partial_count(1), partial_avg(trip_distance#4), partial_avg(fare_amount#10), partial_avg(total_amount#16)])
                              +- *(2) Project [trip_distance#4, fare_amount#10, total_amount#16, Borough#27939]
                   

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

## Where to go from here: answers

### Busiest season of 2019

I download the other 11 months of 2019 (about 1.2 GB) and only read the pickup time. Seasons are the meteorological ones, so winter is January, February and December here. They don't have the same number of days, so I compare trips per day.

In [40]:
df_2019 = None
for month in range(1, 13):
    df_month = load_trips(2019, month).select("tpep_pickup_datetime")
    df_2019 = df_month if df_2019 is None else df_2019.unionByName(df_month)

season = (F.when(F.month("pickup_date").isin(12, 1, 2), "winter")
    .when(F.month("pickup_date").isin(3, 4, 5), "spring")
    .when(F.month("pickup_date").isin(6, 7, 8), "summer")
    .otherwise("fall"))

df_2019_daily = (df_2019
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .filter(F.year("pickup_date") == 2019)
    .groupBy("pickup_date")
    .agg(F.count("*").alias("trips"))
    .withColumn("season", season)
    .cache())

df_seasons = (df_2019_daily
    .groupBy("season")
    .agg(F.sum("trips").alias("trips"),
         F.count("*").alias("days"),
         F.round(F.avg("trips"), 0).alias("avg_trips_per_day"))
    .orderBy(F.desc("avg_trips_per_day")))
df_seasons.show()

(df_2019_daily
    .groupBy(F.month("pickup_date").alias("month"))
    .agg(F.sum("trips").alias("trips"), F.round(F.avg("trips"), 0).alias("avg_trips_per_day"))
    .orderBy("month")
    .show(12))

+------+--------+----+-----------------+
|season|   trips|days|avg_trips_per_day|
+------+--------+----+-----------------+
|spring|22940902|  92|         249358.0|
|winter|21641751|  90|         240464.0|
|  fall|20659549|  91|         227028.0|
|summer|19354800|  92|         210378.0|
+------+--------+----+-----------------+



+-----+-------+-----------------+
|month|  trips|avg_trips_per_day|
+-----+-------+-----------------+
|    1|7696390|         248271.0|
|    2|7049178|         251756.0|
|    3|7866400|         253755.0|
|    4|7475959|         249199.0|
|    5|7598543|         245114.0|
|    6|6971335|         232378.0|
|    7|6310376|         203561.0|
|    8|6073089|         195906.0|
|    9|6567628|         218921.0|
|   10|7214110|         232713.0|
|   11|6877811|         229260.0|
|   12|6896183|         222458.0|
+-----+-------+-----------------+



Spring is the busiest with about 249,000 trips per day, summer the slowest with about 210,000. March is the top month and August the lowest, I guess a lot of people leave the city in summer.

### Visualizations with the native Spark plotting API

Since Spark 4, DataFrame.plot draws charts straight from a Spark DataFrame, using Plotly. The cell installs Plotly if the image doesn't have it.

In [41]:
try:
    import plotly
except ImportError:
    %pip install --quiet plotly
    import plotly
import plotly.io as pio

# plotly_mimetype is rendered by JupyterLab's plotly extension; notebook_connected is an HTML
# fallback (plotly.js loaded from a CDN) for frontends without the extension
pio.renderers.default = "plotly_mimetype+notebook_connected"
print("plotly", plotly.__version__)

plotly 7.1.0


In [42]:
# 1. Average trips per day for each pickup hour (January 2019)
df_hourly.plot.bar(x="pickup_hour", y="avg_trips_per_day",
                   title="January 2019: average trips per day by pickup hour")

In [43]:
# 2. Average trips per day for each day of the week (January 2019)
df_dow_avg.orderBy("pickup_dow").plot.bar(x="pickup_day_name", y="avg_trips_per_day",
                                          title="January 2019: average trips per day by day of the week")

In [44]:
# 3. Average fare by pickup borough (January 2019, boroughs with at least 1,000 trips)
(df_boro_avg
    .filter(F.col("trips") >= 1000)
    .orderBy(F.desc("avg_fare"))
    .plot.bar(x="pu_borough", y="avg_fare", title="January 2019: average fare by pickup borough ($)"))

In [45]:
# 4. Daily trips over 2019
df_2019_daily.orderBy("pickup_date").plot.line(x="pickup_date", y="trips", title="2019: trips per day")

In [46]:
# 5. Average fare, January 2019 vs the recent Januaries
df_compare.withColumn("january", F.col("january").cast("string")).plot.bar(
    x="january", y=["avg_fare", "avg_total"], title="Average fare and total amount paid, by January ($)")

### Another dataset: green taxis, January 2019

Green taxis can't pick up passengers in Manhattan below East 96th / West 110th Street or at the airports, so I wanted to see if the data shows it.

In [47]:
green_file = download("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2019-01.parquet",
                      "green_tripdata_2019-01.parquet")
df_green = add_trip_features(spark.read.parquet(green_file)
    .withColumnRenamed("lpep_pickup_datetime", "tpep_pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "tpep_dropoff_datetime"))
df_green_valid = valid_trips(df_green, 2019, 1)


df_green_pickups = (with_boroughs(df_green_valid)
    .groupBy(F.col("pu_borough").alias("borough"))
    .agg(F.count("*").alias("green_pickups")))

n_green_valid = df_green_valid.count()
print(f"green taxis, January 2019: {df_green.count():,} records, {n_green_valid:,} valid trips "
      f"(yellow: {n_valid:,})")
(df_pickups
    .join(df_green_pickups, "borough", "full")
    .fillna(0)
    .withColumn("yellow_pct", F.round(F.col("pickups") / n_valid * 100, 1))
    .withColumn("green_pct", F.round(F.col("green_pickups") / n_green_valid * 100, 1))
    .orderBy(F.desc("green_pickups"))
    .show())

summary_metrics(df_green_valid, 2019).drop("january").show()

green taxis, January 2019: 672,105 records, 650,190 valid trips (yellow: 7,583,742)


+-------------+-------+-------------+----------+---------+
|      borough|pickups|green_pickups|yellow_pct|green_pct|
+-------------+-------+-------------+----------+---------+
|     Brooklyn|  88292|       209630|       1.2|     32.2|
|    Manhattan|6876789|       202611|      90.7|     31.2|
|       Queens| 450841|       183590|       5.9|     28.2|
|        Bronx|  16850|        53628|       0.2|      8.2|
|Staten Island|    297|          305|       0.0|      0.0|
|      Unknown| 149052|          285|       2.0|      0.0|
|          N/A|   1591|          139|       0.0|      0.0|
|          EWR|     30|            2|       0.0|      0.0|
+-------------+-------+-------------+----------+---------+



+------+--------------+---------------+----------------+-------------+--------+------------+----------------+---------+----------------------+------------------+
| trips|avg_passengers|avg_distance_mi|avg_duration_min|avg_speed_mph|avg_fare|avg_tip_card|avg_tip_pct_card|avg_total|card_pct_known_payment|payment_type_0_pct|
+------+--------------+---------------+----------------+-------------+--------+------------+----------------+---------+----------------------+------------------+
|650190|         1.321|            3.9|           16.49|         13.0|   15.89|         1.4|            13.0|    18.25|                  62.5|               0.0|
+------+--------------+---------------+----------------+-------------+--------+------------+----------------+---------+----------------------+------------------+



In [48]:
# Where are the green pickups located in Manhattan?
(with_boroughs(df_green_valid)
    .filter(F.col("pu_borough") == "Manhattan")
    .groupBy("pu_zone")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False))

+------------------------+-----+
|pu_zone                 |count|
+------------------------+-----+
|East Harlem North       |42480|
|East Harlem South       |39916|
|Central Harlem          |32966|
|Morningside Heights     |21900|
|Central Harlem North    |20743|
|Washington Heights South|15545|
|Hamilton Heights        |8565 |
|Central Park            |7116 |
|Manhattanville          |4135 |
|Washington Heights North|3366 |
+------------------------+-----+
only showing top 10 rows


Yellow cabs do 11.7 times more trips, but green ones work somewhere else: their pickups are spread over Brooklyn (32%), Manhattan (31%), Queens (28%) and the Bronx (8%), and the Manhattan ones are mostly in upper Manhattan (Harlem, Washington Heights). Their trips are longer (3.9 vs 2.85 miles) and the card tips lower (13% vs 21.8%).

In [49]:
spark.stop()